In [ ]:
import numpy as np
import ufl

from mpi4py import MPI

import dolfinx
import dolfinx.fem.petsc
import dolfinx.mesh
import basix.ufl
from petsc4py import PETSc

N = 2

domain = dolfinx.mesh.create_unit_square(
    MPI.COMM_WORLD, 
    N, N, 
    dolfinx.mesh.CellType.quadrilateral
)
coords = domain.geometry.x
xi = coords[:, 0]   # x-coordinates of the unit square
eta = coords[:, 1]  # y-coordinates of the unit square
x_new = 4.8 * xi
y_new = 4.4 * eta + 4.4 * xi - 2.8 * xi * eta
domain.geometry.x[:, 0] = x_new
domain.geometry.x[:, 1] = y_new


dim = domain.topology.dim
print(f"Mesh topology dimension d={dim}.")

degree = 2
#shape = (dim,)  # this means we want a vector field of size `dim`
v_elem = basix.ufl.element(
    "Lagrange", 
    domain.topology.cell_name(), 
    degree, 
    shape=(dim,)
)
V = dolfinx.fem.functionspace(domain, v_elem)

u_sol = dolfinx.fem.Function(V, name="Displacement")

E = dolfinx.fem.Constant(domain, 70.)
nu = dolfinx.fem.Constant(domain, 1./3.)

lmbda = E * nu / (1. + nu) / (1. - 2. * nu)
mu = E / 2. / (1. + nu)

def right_boundary(x):
    return np.isclose(x[0], 4.8)
def left_boundary(x):
    return np.isclose(x[0], 0.)

facet_dim = domain.topology.dim-1
boundary_facets = dolfinx.mesh.locate_entities_boundary(domain, facet_dim, right_boundary)
boundary_tags = dolfinx.mesh.meshtags(
    domain,
    facet_dim,
    boundary_facets,
    np.full(len(boundary_facets), 1, dtype=np.int32)
)

def epsilon(v):
    return ufl.sym(ufl.grad(v))


def sigma(v):
    return lmbda * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu * epsilon(v)

u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

T = dolfinx.fem.Constant(
    domain, 
    dolfinx.default_scalar_type((0.0, 6.25))
)


custom_metadata = {"quadrature_degree": 8}
custom_dx = ufl.Measure("dx", domain=domain, metadata=custom_metadata)
custom_ds = ufl.Measure("ds", domain=domain, subdomain_data=boundary_tags, metadata=custom_metadata)
a = ufl.inner(sigma(u), epsilon(v)) * custom_dx
L = ufl.inner(T, v)*custom_ds(1)

left_dofs = dolfinx.fem.locate_dofs_geometrical(V, left_boundary)
zero_vec = np.zeros(dim, dtype=dolfinx.default_scalar_type)
bcs = [
    dolfinx.fem.dirichletbc(zero_vec, left_dofs, V),
]
problem = dolfinx.fem.petsc.LinearProblem(
    a, L, u=u_sol, bcs=bcs,
    petsc_options_prefix="cooks_membrane",
    petsc_options={
        "ksp_type": "preonly", 
        "pc_type": "lu",
        "pc_factor_mat_solver_type": "mumps"}
)
problem.solve()
# A = dolfinx.fem.petsc.assemble_matrix(a, bcs=bcs)
# A.assemble()

# # Assemble b
# b = dolfinx.fem.petsc.assemble_vector(L)
# dolfinx.fem.petsc.apply_lifting(b, [a], bcs=[bcs])
# b.ghostUpdate(addv=PETSc.InsertMode.ADD_VALUES,
#               mode=PETSc.ScatterMode.REVERSE)
# dolfinx.fem.petsc.set_bc(b, bcs)
#b_lagrange = problem.b
#A_lagrange = problem.A

In [ ]:
import pyvista
from dolfinx.plot import vtk_mesh

topology, cell_types, geometry = vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# The solution array is 1D. We reshape it to N x 2 (since dim=2)
u_values = u_sol.x.array.reshape(-1, dim)

# PyVista requires 3D vectors to warp the mesh. We pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Warp the grid by the displacement. 
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# Plotting
plotter = pyvista.Plotter()
plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the original undeformed mesh as a wireframe
plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# Show the deformed mesh
plotter.add_mesh(warped_grid, show_edges=False, scalars="Displacement", cmap="coolwarm", label="Deformed")

plotter.view_xy()  # Set camera to view the X-Y plane directly
plotter.show(jupyter_backend="static")

In [ ]:
print(f"Top right corner vertical displacement: {np.max(u_3d[:,1])*10:.2f}mm")

In [ ]:
from THBSplines.src.hierarchical_space import HierarchicalSpace
from THBSplines.src.cartesian_mesh import CartesianMesh
import numpy as np
import scipy.sparse as sp
import dolfinx
from mpi4py import MPI
import basix.ufl
import pyvista

import cffi
import numba
import numba.core.typing.cffi_utils as cffi_support
from dolfinx.jit import ffcx_jit
from dolfinx import default_real_type, default_scalar_type, geometry
rtype = default_real_type
dtype = default_scalar_type
import ufl
from ffcx.codegeneration.utils import empty_void_pointer
from ffcx.codegeneration.utils import numba_ufcx_kernel_signature as ufcx_signature

import numpy.typing as npt


from solve_utils import refine, build_mesh_2d, build_dofmap, fill_function_space, create_spline_space

In [ ]:
n_refinements = 1
p0 = 2
knotsx = np.array([0,0,0, 0.5,1, 1, 1], dtype=np.float64)
knotsx = refine(knotsx, p=p0, n_times=n_refinements)
log_initial_mesh_size = np.log2(np.max(np.diff(knotsx)))
knotsy = np.array([0,0,0,0.5,1,1,1], dtype=np.float64)
knotsy = refine(knotsy, p0, n_times=n_refinements)
err_cells = {}
hs = HierarchicalSpace(knots=[knotsx, knotsy], degrees=[p0])

In [ ]:
for level, cells in err_cells.items():
    hs.refine(cells, level, refine_neighbours=False)
hs.hmesh.plot_cells()

In [ ]:
def mapping_to_trapezoid(uv_points):
    original_shape = uv_points.shape
    uv_flat = uv_points.reshape(-1, 2)
    xy_flat = np.zeros_like(uv_flat)

    def bilinear(u_loc, v_loc, C00, C10, C01, C11):
        """Standard isoparametric Q1 interpolation"""
        return ((1-u_loc)*(1-v_loc)*C00 + 
                u_loc*(1-v_loc)*C10 + 
                (1-u_loc)*v_loc*C01 + 
                u_loc*v_loc*C11)
    
    P00 = np.array([0.0, 0.0])
    P10 = np.array([4.8, 4.4])
    P01 = np.array([0.0, 4.4])
    P11 = np.array([4.8, 6.0])

    xy_flat[:, 0] = bilinear(u_loc=uv_flat[:, 0], v_loc=uv_flat[:, 1],
                             C00=P00[0], C10=P10[0], C01=P01[0], C11=P11[0])
    xy_flat[:, 1] = bilinear(u_loc=uv_flat[:, 0], v_loc=uv_flat[:, 1],
                             C00=P00[1], C10=P10[1], C01=P01[1], C11=P11[1])
    
    
    return xy_flat.reshape(original_shape)

disconnected_mesh, thb_operators, N_max, trapez_midpoints = build_mesh_2d(hs=hs, mapping=mapping_to_trapezoid)

In [ ]:
topology, cell_types, geometry = dolfinx.plot.vtk_mesh(disconnected_mesh)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

plotter = pyvista.Plotter()
plotter.add_mesh(grid.shrink(.95), show_edges=False, color="#03bb85")
plotter.view_xy()
plotter.show(jupyter_backend="static")
print(f"Number of points in PyVista grid: {grid.n_points}")

In [ ]:
tdim = disconnected_mesh.topology.dim
fdim = tdim - 1
disconnected_mesh.topology.create_connectivity(fdim, tdim)

coords = disconnected_mesh.geometry.x
# midpoints = np.mean(coords.reshape(-1, 4, 2), axis=1)
x_min = np.min(coords[:, 0])  # Find the leftmost x-value dynamically
x_max = np.max(coords[:, 0])
y_min = np.min(coords[:, 1])
y_max = np.max(coords[:, 1])
y_min2 = np.min(coords[:, 1][np.isclose(coords[:, 0], x_max)])
y_max2 = np.max(coords[:, 1][np.isclose(coords[:, 0], x_min)])

T00 = np.array([x_min, y_min], dtype=float)
T01 = np.array([x_max, y_min2], dtype=float)
T10 = np.array([x_min, y_max2], dtype=float)
T11 = np.array([x_max, y_max], dtype=float)

def left_boundary(x):
    return np.isclose(x[0], x_min)
def right_boundary(x):
    return np.isclose(x[0], x_max)
def bottom_boundary(x):
    return np.isclose(x[1], x[0]*T01[1]/T01[0])
def top_boundary(x):
    return np.isclose(x[1], T10[1]+x[0]*(T11[1]-T10[1])/T11[0])
def bottom_right_top_boundary(x):
    return right_boundary(x)|bottom_boundary(x)|top_boundary(x)
def not_right(x):
    return ~right_boundary(x)


In [ ]:
left_facets = dolfinx.mesh.locate_entities_boundary(
    disconnected_mesh, 
    fdim, 
    left_boundary
)

right_facets = dolfinx.mesh.locate_entities_boundary(
    disconnected_mesh, 
    fdim, 
    right_boundary
)

#sorted_facets = np.sort(right_facets)

# Assign a marker ID (e.g., 1) to these facets
facet_values = np.full_like(right_facets, 1, dtype=np.int32)

# Create the MeshTags object
facet_tags = dolfinx.mesh.meshtags(
    disconnected_mesh, fdim, right_facets, facet_values
)

dim = disconnected_mesh.topology.dim
legendre_elt = basix.ufl.element(
    "DG",
    "quadrilateral",
    degree=p0,
    shape=(dim,),
    lagrange_variant=basix.LagrangeVariant.legendre
)
V = dolfinx.fem.functionspace(disconnected_mesh, legendre_elt)
# print(f"Number of degrees of freedom: {V.dofmap.index_map.size_global}")

#boundary_facets = dolfinx.mesh.locate_entities_boundary(disconnected_mesh, facet_dim, Gamma_N)
custom_metadata = {"quadrature_degree": 6}
ds_custom = ufl.Measure("ds", domain=disconnected_mesh, subdomain_data=facet_tags, metadata=custom_metadata)
dx_custom = ufl.Measure("dx", domain=disconnected_mesh, metadata=custom_metadata)

E=70.0
nu = 1./3.
lmbda = E*nu/(1.+nu)/(1.-2.*nu)
#mu = E / 2. / (1. + nu)
mu = E/2./(1.+nu)
lmbda_c = dolfinx.fem.Constant(disconnected_mesh, lmbda)
mu_c = dolfinx.fem.Constant(disconnected_mesh, mu)

def epsilon(v):
    return ufl.sym(ufl.grad(v))

def sigma(v):
    return lmbda_c * ufl.tr(epsilon(v)) * ufl.Identity(dim) + 2. * mu_c * epsilon(v)


u,v = ufl.TrialFunction(V), ufl.TestFunction(V) 
T = dolfinx.fem.Constant(
    disconnected_mesh, 
    dolfinx.default_scalar_type((0., 6.25))
)
a = ufl.inner(sigma(u),epsilon(v))*dx_custom
L_facet = ufl.inner(T, v)*ds_custom(1)

msh = disconnected_mesh
ufcxa0, _, _ = ffcx_jit(msh.comm, a, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernela0 = getattr(ufcxa0.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ufcx_L_facet, _, _ = ffcx_jit(msh.comm, L_facet, form_compiler_options={"scalar_type": dtype})  # type: ignore
kernel_L_facet = getattr(ufcx_L_facet.form_integrals[0], f"tabulate_tensor_{np.dtype(dtype).name}")  # type: ignore

ffi = cffi.FFI()

In [ ]:
# import pyvista
# from dolfinx import plot

# # 1. Extract the mesh geometry of ONLY the right facets
# topology_f, cell_types_f, geometry_f = plot.vtk_mesh(disconnected_mesh, fdim, right_facets)
# facet_grid = pyvista.UnstructuredGrid(topology_f, cell_types_f, geometry_f)

# # 2. Extract the full mesh for context
# topology_m, cell_types_m, geometry_m = plot.vtk_mesh(disconnected_mesh, tdim)
# mesh_grid = pyvista.UnstructuredGrid(topology_m, cell_types_m, geometry_m)

# # 3. Plot them together
# plotter = pyvista.Plotter()

# # Plot full mesh as a wireframe
# plotter.add_mesh(mesh_grid, style="wireframe", color="black", line_width=1)

# # Plot right boundary facets as a thick red line
# plotter.add_mesh(facet_grid, color="red", line_width=5, render_lines_as_tubes=True)

# plotter.view_xy()
# plotter.show(jupyter_backend="static")

In [ ]:
dofmap, padded_cells_to_dofs = build_dofmap(hierarchical_space=hs, mesh=disconnected_mesh, N_max=N_max, morton=False)

In [ ]:
#trapez_midpoints

In [ ]:
def find_level_idx_from_midpoint(hs, midpoint, all_midpoints):
    truth_array = np.all(np.isclose(midpoint,all_midpoints), axis=-1)
    global_index = np.nonzero(truth_array)[0][0]
    level=0
    current_elts = hs.hmesh.aelem_level[level]
    #print(f"\nglobal_index = {global_index}")
    while(global_index>len(current_elts)-1):
        global_index-=len(current_elts)
        level+=1
        current_elts = hs.hmesh.aelem_level[level]
        #print(f"in for loop, global_index = {global_index}")
    #print(f"\n")
    return (level, hs.hmesh.aelem_level[level][global_index])

custom_mapping = lambda pt: find_level_idx_from_midpoint(hs=hs, midpoint=pt, all_midpoints=trapez_midpoints)
C_func, C_space = fill_function_space(hierachical_space=hs, mesh=disconnected_mesh, N_max = N_max, 
                                      thb_operators=thb_operators, mapping_function=custom_mapping)

In [ ]:
V_spline = create_spline_space(cells_to_dofs=padded_cells_to_dofs, mesh=disconnected_mesh,
                               N_max=N_max, mult_factor=2)

In [ ]:
PADDED_DOFS = N_max
LOCAL_DOFS = (hs.degrees[0]+1)**2

LOCAL_DOFS_VEC = 2 * (hs.degrees[0]+1)**2
PADDED_DOFS_VEC = 2 * N_max

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)  # type: ignore
def tabulate_A(A_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):

    # Prepare target condensed local element tensor
    # arguments: ptr, shape, dtype
    # returns a view over the original array A_
    # This has to be larger since we are working with padded arrays. Irrelevant dofs are mapped to a dummy location
    A = numba.carray(A_, (PADDED_DOFS_VEC, PADDED_DOFS_VEC), dtype=dtype)

    # Get the operator (TRUNC @ C_{B->BS} @ (C_{L->B}.T) @ S^{-1}) for this cell
    # TRUNC has shape (PADDED_DOFS, LOCAL_DOFS) and all other matrices have shape (LOCAL_DOFS, LOCAL_DOFS)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)
    
    # Tabulate all sub blocks locally
    # This matrix is formed via the Legendre elements on a quadrilateral of degree p0,
    # therefore this has the shape (LOCAL_DOFS, LOCAL_DOFS)
    A0 = np.zeros((LOCAL_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    kernela0(
        ffi.from_buffer(A0),
        w_,
        c_,
        coords_,
        entity_local_index,
        permutation,
        empty_void_pointer(),
    )

    G_vec = np.zeros((PADDED_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    for i in range(PADDED_DOFS):
        for j in range(LOCAL_DOFS):
            G_vec[2*i, 2*j]     = G[i, j] # X-component mapping
            G_vec[2*i+1, 2*j+1] = G[i, j] # Y-component mapping
        pass
    pass
            
    
    A[:, :] = G_vec@A0@(G_vec.T) 

@numba.cfunc(ufcx_signature(dtype, rtype), nopython=True)
def tabulate_L_facet(b_, w_, c_, coords_, entity_local_index, permutation=ffi.NULL, custom_data=None):
    b = numba.carray(b_, (PADDED_DOFS_VEC,), dtype=dtype)
    G = numba.carray(w_, (PADDED_DOFS, LOCAL_DOFS), dtype=dtype)

    b0 = np.zeros((LOCAL_DOFS_VEC,), dtype=dtype)
    kernel_L_facet(ffi.from_buffer(b0), 
                   w_, 
                   c_, 
                   coords_, 
                   entity_local_index, 
                   permutation, 
                   empty_void_pointer())
   
    
    G_vec = np.zeros((PADDED_DOFS_VEC, LOCAL_DOFS_VEC), dtype=dtype)
    for i in range(PADDED_DOFS):
        for j in range(LOCAL_DOFS):
            G_vec[2*i, 2*j]     = G[i, j]
            G_vec[2*i+1, 2*j+1] = G[i, j]
            
    b[:] = G_vec @ b0

In [ ]:
facet_dim = msh.topology.dim-1

boundary_facets = dolfinx.mesh.locate_entities_boundary(msh, facet_dim, right_boundary)
msh.topology.create_connectivity(facet_dim, msh.topology.dim)
msh.topology.create_connectivity(msh.topology.dim, facet_dim)

# dictionary of facet->cell
f_to_c = msh.topology.connectivity(facet_dim, msh.topology.dim)
# dictionary of cell->array[facets]
c_to_f = msh.topology.connectivity(msh.topology.dim, facet_dim)

boundary_entities = []
# Loop over all edges that belong to our exterior
for f in boundary_facets:
    # returns the cells linked to this edge
    cells = f_to_c.links(f)
    # Exterior facets only have 1 attached cell, 
    # hence get the first one 
    c = cells[0] 
    
    # Find the local index (e.g., 0, 1, 2, or 3 for quads) of facet f within cell c
    local_f = np.where(c_to_f.links(c) == f)[0][0]
    #print(f"f={f}, cell={c}, local_f = {local_f}")
    
    boundary_entities.extend([c, local_f])

# FEniCSx custom form arrays must be typed as int32
boundary_entities = np.array(boundary_entities, dtype=np.int32)

In [ ]:
# print(boundary_entities.reshape(-1, 2))

In [ ]:
lhs_constants = ufl.algorithms.analysis.extract_constants(a)
# Extract the underlying C++ objects in that exact order
cpp_constants_lhs = [c._cpp_object for c in lhs_constants]

rhs_constants = ufl.algorithms.analysis.extract_constants(L_facet)
cpp_constants_rhs = [c._cpp_object for c in rhs_constants]

formtype = dolfinx.fem.form_cpp_class(dtype)  # type: ignore
# Gets the number of cells for which each individual core is responsible for.
cells = np.arange(msh.topology.index_map(msh.topology.dim).size_local, dtype=np.int32)

# The 4th argument np.array([...], dtype=np.int8) is the 
# active coefficients array. It lists which indices from the 
# coefficients list should be packed into the w_ pointer that the kernel receives.
integrals = {dolfinx.fem.IntegralType.cell: [
    (0, tabulate_A.address, cells, np.array([0], dtype=np.int8))]}

a_cond = dolfinx.fem.Form( # We are not forming anything yet, this is a recipe
    formtype( # selectes the correct floating-point precision
        spaces=[V_spline._cpp_object, 
                V_spline._cpp_object]
            , # trial and test spaces, determines the size of A_
        integrals=integrals, #this is a dictionary, and we are passing the adress of tabulate_A() here
        coefficients=[C_func._cpp_object], # weights w_, holds C@T
              constants=cpp_constants_lhs,
              need_permutation_data=False,
              entity_maps=[], 
              mesh=msh._cpp_object)
)

integrals_rhs = {dolfinx.fem.IntegralType.exterior_facet: [(0, tabulate_L_facet.address, boundary_entities, np.array([0], dtype=np.int8))]}
l_cond = dolfinx.fem.Form(
    formtype(
        spaces=[V_spline._cpp_object], # test space, determines the size of b_
        integrals=integrals_rhs, #give the adress of stuff to integrate
        coefficients=[C_func._cpp_object], # holds C@T
        constants=cpp_constants_rhs, 
        need_permutation_data=False, 
        entity_maps=[], 
        mesh=msh._cpp_object
    )
)

In [ ]:
def get_spline_indices_left(hs, dofmap):
    dirichlet_indices = {}
    for level in range(hs.nlevels):
        hs.level_spaces[level].construct_basis()
        basis = hs.level_spaces[level].basis
        
        left_dirichlet_indices = np.isclose(basis[:, 1, :-1], np.zeros((hs.degrees[0]+1), dtype=np.float64))
        left_dirichlet_indices = np.all(left_dirichlet_indices, axis=-1)
        left_dirichlet_indices = np.nonzero(left_dirichlet_indices)[0]

        dirichlet_indices[level] = np.intersect1d(hs.truly_active[level],
                                                  left_dirichlet_indices,
                                                  assume_unique=True)
    pass
    forbidden_indices = np.array([dofmap[level,idx] for level in dirichlet_indices 
              for idx in dirichlet_indices[level]],
                        dtype=np.int32)
    return forbidden_indices

forbidden_indices = get_spline_indices_left(hs, dofmap)
if forbidden_indices is not None and len(forbidden_indices) > 0:
    forbidden_indices_vec = np.empty(2 * len(forbidden_indices), dtype=np.int32)
    forbidden_indices_vec[0::2] = 2 * forbidden_indices      # X DOFs
    forbidden_indices_vec[1::2] = 2 * forbidden_indices + 1  # Y DOFs
    forbidden_indices = forbidden_indices_vec

In [ ]:
# forbidden_indices

In [ ]:
from dolfinx.fem.petsc import assemble_matrix, assemble_vector
from petsc4py import PETSc
A = assemble_matrix(a_cond, bcs=[])
A.assemble()
one_active=False
two_active= False
for level in range(hs.nlevels):
    if level in hs.truly_active and hs.truly_active[level].size>0:
        if one_active:
            two_active=True
        one_active=True

A_mat = A
if two_active:
    dummy_dof_index = np.max(padded_cells_to_dofs)
    dummy_x = 2 * dummy_dof_index
    dummy_y = 2 * dummy_dof_index + 1
    A_mat.setValue(dummy_x, dummy_x, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.setValue(dummy_y, dummy_y, 1., addv=PETSc.InsertMode.INSERT_VALUES)
    A_mat.assemble()
    A_mat.assemblyBegin()
    A_mat.assemblyEnd()

b = assemble_vector(l_cond)
if two_active:
    b[dummy_x] = 0.0
    b[dummy_y] = 0.0
    b.assemblyBegin()
    b.assemblyEnd()

if forbidden_indices is not None:
    A_mat.zeroRowsColumns(forbidden_indices, diag=1.0, x=None, b=b)
    b.array_w[forbidden_indices]=0.
b.ghostUpdate(addv=PETSc.InsertMode.INSERT, mode=PETSc.ScatterMode.FORWARD)

In [ ]:
ksp = PETSc.KSP().create(A_mat.comm)
ksp.setOperators(A_mat)
#ksp.setType(PETSc.KSP.Type.CG)
#ksp.getPC().setType(PETSc.PC.Type.JACOBI)
ksp.setType(PETSc.KSP.Type.PREONLY)
ksp.getPC().setType(PETSc.PC.Type.LU)
ksp.getPC().setFactorSolverType("mumps")
u_sol = dolfinx.fem.Function(V_spline)
ksp.solve(b, u_sol.x.petsc_vec)
u_sol.x.scatter_forward()
x_vec=u_sol.x.array
print(f"Solve complete. Reason: {ksp.getConvergedReason()}, Iterations: {ksp.getIterationNumber()}")

In [ ]:
# x_vec.shape

In [ ]:
u_dg = dolfinx.fem.Function(V)
block_size = V.dofmap.bs
c_values = C_func.x.array.reshape((-1, N_max, (hs.degrees[0]+1)**2))
vector_dofs = np.zeros((padded_cells_to_dofs.shape[0], 2*N_max), dtype=np.int32)
for i in range(2):
    vector_dofs[:, i::2]=2*padded_cells_to_dofs+i

for local_idx in range(msh.topology.index_map(msh.topology.dim).size_local):
    spline_dofs = vector_dofs[local_idx]
    u_spline_local = x_vec[spline_dofs]
    
    G = c_values[local_idx, :, :]
    G_vec = np.zeros((2*G.shape[0], 2*G.shape[1]), dtype=dtype)
    for i in range(G.shape[0]):
        for j in range(G.shape[1]):
            G_vec[2*i, 2*j]     = G[i, j] # X-component mapping
            G_vec[2*i+1, 2*j+1] = G[i, j] # Y-component mapping
        pass
    pass
    u_dg_local = G_vec.T @ u_spline_local
    
    dg_dofs = V.dofmap.cell_dofs(local_idx)
    unrolled_dg_dofs = np.empty(len(dg_dofs) * block_size, dtype=np.int32)
    for i in range(block_size):
        unrolled_dg_dofs[i::block_size] = dg_dofs * block_size + i
        
    u_dg.x.array[unrolled_dg_dofs] = u_dg_local

In [ ]:
print(f"Top right corner vertical displacement: {np.max(u_dg.x.array.reshape(-1, 2)[:,1])*10:.2f}mm")

In [ ]:
import pyvista
import numpy as np
import dolfinx
from dolfinx.plot import vtk_mesh

dim = msh.geometry.dim

# 1. Create a plot-friendly function space: Continuous Galerkin (Lagrange) degree 1
# This guarantees that DOFs match the physical vertices of the mesh.
V_plot = dolfinx.fem.functionspace(msh, ("Lagrange", 1, (dim,)))

# 2. Interpolate your Legendre/DG function into this nodal space
u_plot = dolfinx.fem.Function(V_plot)
u_plot.interpolate(u_dg)

# 3. Generate the VTK mesh from the plotting space
topology, cell_types, geometry = vtk_mesh(V_plot)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# 4. Extract the solution array and reshape it to N x dim
# Because V_plot is a vector space, DOFs are interleaved (x0, y0, x1, y1...)
u_values = u_plot.x.array.reshape(-1, dim)

# 5. PyVista requires 3D vectors to warp the mesh. Pad the 2D vectors with Z=0.
u_3d = np.zeros((u_values.shape[0], 3), dtype=np.float64)
u_3d[:, :dim] = u_values

# 6. Attach the 3D displacement vectors to the grid
grid.point_data["Displacement"] = u_3d
grid.set_active_vectors("Displacement")

# Optional: Calculate displacement magnitude to use for coloring
grid.point_data["Displacement_Magnitude"] = np.linalg.norm(u_3d, axis=1)

# 7. Warp the grid by the displacement
warp_factor = 1.0
warped_grid = grid.warp_by_vector("Displacement", factor=warp_factor)

# 8. Plotting
plotter = pyvista.Plotter()
plotter.add_text(f"Deformed Mesh (magnified {warp_factor}x)", font_size=14)

# Show the original undeformed mesh as a wireframe
plotter.add_mesh(grid, style="wireframe", color="black", opacity=0.3, label="Undeformed")

# Show the deformed mesh, coloring it by the magnitude of the displacement
plotter.add_mesh(
    warped_grid, 
    show_edges=False, 
    scalars="Displacement_Magnitude", # Color by magnitude instead of the vector array
    cmap="coolwarm", 
    label="Deformed"
)

plotter.view_xy()  # Set camera to view the X-Y plane directly
plotter.show(jupyter_backend="static")